In [1]:
!pip install scikit-learn==1.5.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 96.2 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tpot 1.1.0 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
category-encoders 2.9.0 requires scikit-learn>=1.6.0, but you have scikit-learn 1.5.2 which is incompatible.
umap-learn 0.5.11 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.41 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [2]:
import xgboost as xgb
import pandas as pd
import numpy as np
import warnings
import optuna
import gc
import os
from sklearn.preprocessing import TargetEncoder, StandardScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold, KFold
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import balanced_accuracy_score
from sklearn.decomposition import PCA
from optuna.samplers import TPESampler
from itertools import combinations
from xgboost import XGBClassifier
from tqdm import tqdm

warnings.filterwarnings("ignore", category=pd.errors.ChainedAssignmentError)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)
warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
TARGET = 'Irrigation_Need'
NUMS = ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']
CATS = ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']

def metric(y_true, y_pred):
    y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)

In [3]:
train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/train.csv', index_col='id')
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/test.csv', index_col='id')
orig = pd.read_csv('/kaggle/input/datasets/miadul/irrigation-water-requirement-prediction-dataset/irrigation_prediction.csv')
train[TARGET] = train[TARGET].map({'Low': 0, 'Medium': 1, 'High': 2})
orig[TARGET] = orig[TARGET].map({'Low': 0, 'Medium': 1, 'High': 2})

combined = pd.concat([train, test, orig])
for c in CATS:
    combined[c], _ = combined[c].factorize()
combined[CATS] = combined[CATS].astype('category')
train = combined[:len(train)]
test = combined[len(train):len(train) + len(test)].drop(TARGET, axis=1)
orig = combined[len(train) + len(test):]

TE_columns = []

columns = NUMS + CATS

for r in [2]:
    for cols in tqdm(list(combinations(columns, r))):
        name = '-'.join(cols)

        train[name] = train[cols[0]].astype(str)
        for col in cols[1:]:
            train[name] = train[name] + '_' + train[col].astype(str)

        test[name] = test[cols[0]].astype(str)
        for col in cols[1:]:
            test[name] = test[name] + '_' + test[col].astype(str)

        orig[name] = orig[cols[0]].astype(str)
        for col in cols[1:]:
            orig[name] = orig[name] + '_' + orig[col].astype(str)

        combined = pd.concat([train[name], test[name], orig[name]], ignore_index=True)
        combined, _ = combined.factorize()
        if pd.Series(combined).nunique() > len(combined) // 2:
            train = train.drop(name, axis=1)
            test = test.drop(name, axis=1)
            orig = orig.drop(name, axis=1)
            continue
        train[name] = combined[:len(train)]
        test[name] = combined[len(train):len(train) + len(test)]
        orig[name] = combined[len(train) + len(test):]

        TE_columns.append(name)

TE_ORIG = []
CC = CATS + NUMS

print(f"Processing {len(CC)} columns... ",end="")
for i,c in enumerate(CC):
    #if train[c].nunique() == 2:
    #    continue
    if i%10==0: print(f"{i}, ",end="")
    tmp = orig.groupby(c)[TARGET].mean()
    tmp = tmp.astype('float32')
    tmp.name = f"TE_ORIG_{c}"
    TE_ORIG.append( f"TE_ORIG_{c}" )
    train = train.merge(tmp, on=c, how='left')
    train[tmp.name] = train[tmp.name].fillna(0.5)
    test = test.merge(tmp, on=c, how='left')
    test[tmp.name] = test[tmp.name].fillna(0.5)
print()

FEATURES = train.columns.tolist()
FEATURES.remove(TARGET)

100%|██████████| 171/171 [02:26<00:00,  1.17it/s]


Processing 19 columns... 0, 10, 


In [4]:
def balanced_accuracy():
    def xgb_balanced_accuracy(y_true, y_pred):
        n_classes = 3
        y_pred_labels = np.argmax(y_pred.reshape(-1, n_classes), axis=1)
        return balanced_accuracy_score(y_true.astype(int), y_pred_labels)

    xgb_balanced_accuracy.__name__ = 'bal_ACC'
    return xgb_balanced_accuracy

In [5]:
oof = np.zeros((len(train), 3))
pred = np.zeros((len(test), 3))

skf = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)

for idx, (train_idx, val_idx) in enumerate(skf.split(np.zeros(len(train)), train[TARGET])):
    X_train, X_val = train.loc[train_idx, FEATURES], train.loc[val_idx, FEATURES]
    y_train, y_val = train.loc[train_idx, TARGET], train.loc[val_idx, TARGET]
    X_test = test.copy()

    encoder = TargetEncoder(target_type='multiclass', cv=5, random_state=42)
    result = pd.DataFrame(encoder.fit_transform(X_train[TE_columns], y_train))
    X_train = pd.concat([result.reset_index(drop=True), X_train.reset_index(drop=True)], axis=1)
    result = pd.DataFrame(encoder.transform(X_val[TE_columns]))
    X_val = pd.concat([result.reset_index(drop=True), X_val.reset_index(drop=True)], axis=1)
    result = pd.DataFrame(encoder.transform(X_test[TE_columns]))
    X_test = pd.concat([result.reset_index(drop=True), X_test.reset_index(drop=True)], axis=1)
    X_train = X_train.drop(TE_columns, axis=1)
    X_val = X_val.drop(TE_columns, axis=1)
    X_test = X_test.drop(TE_columns, axis=1)

    classes = np.unique(y_train)
    weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weight_dict = dict(zip(classes, weights))
    sample_weights = np.array([class_weight_dict[label] for label in y_train])

    param_grid = {'max_depth': 6,
                  'subsample': 0.8,
                  'colsample_bytree': 0.8}

    model = XGBClassifier(**param_grid,
                          n_estimators=50000,
                          objective='multi:softprob',
                          learning_rate=0.01,
                          eval_metric=balanced_accuracy(),
                          #grow_policy='lossguide',
                          callbacks=[
                              xgb.callback.EarlyStopping(
                                  rounds=500, metric_name='bal_ACC',
                                  maximize=True, save_best=True
                              )
                          ],
                          max_bin=1024,
                          random_state=42,
                          enable_categorical=True,
                          device='cuda',
                          n_jobs=-1)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], sample_weight=sample_weights, verbose=100)
    oof[val_idx] = model.predict_proba(X_val)
    pred += model.predict_proba(X_test)
    
    print(f'Fold {idx + 1}: {metric(y_val, oof[val_idx])}')

    if idx < 4: del model
    del X_train, X_val, y_train, y_val
    gc.collect()

pred /= 5
print(f'CV ACC: {metric(train[TARGET], oof)}')

[0]	validation_0-mlogloss:1.08591	validation_0-bal_ACC:0.95566
[100]	validation_0-mlogloss:0.41355	validation_0-bal_ACC:0.97015
[200]	validation_0-mlogloss:0.20608	validation_0-bal_ACC:0.97202
[300]	validation_0-mlogloss:0.12636	validation_0-bal_ACC:0.97300
[400]	validation_0-mlogloss:0.09153	validation_0-bal_ACC:0.97409
[500]	validation_0-mlogloss:0.07565	validation_0-bal_ACC:0.97481
[600]	validation_0-mlogloss:0.06720	validation_0-bal_ACC:0.97516
[700]	validation_0-mlogloss:0.06231	validation_0-bal_ACC:0.97537
[800]	validation_0-mlogloss:0.05927	validation_0-bal_ACC:0.97554
[900]	validation_0-mlogloss:0.05713	validation_0-bal_ACC:0.97559
[1000]	validation_0-mlogloss:0.05547	validation_0-bal_ACC:0.97573
[1100]	validation_0-mlogloss:0.05423	validation_0-bal_ACC:0.97584
[1200]	validation_0-mlogloss:0.05329	validation_0-bal_ACC:0.97595
[1300]	validation_0-mlogloss:0.05253	validation_0-bal_ACC:0.97598
[1400]	validation_0-mlogloss:0.05191	validation_0-bal_ACC:0.97598
[1500]	validation_0-ml

/usr/local/lib/python3.12/dist-packages/xgboost/core.py:751: UserWarning: [10:59:20] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Fold 1: 0.976112566613836
[0]	validation_0-mlogloss:1.08594	validation_0-bal_ACC:0.95671
[100]	validation_0-mlogloss:0.41624	validation_0-bal_ACC:0.97114
[200]	validation_0-mlogloss:0.20657	validation_0-bal_ACC:0.97249
[300]	validation_0-mlogloss:0.12681	validation_0-bal_ACC:0.97352
[400]	validation_0-mlogloss:0.09218	validation_0-bal_ACC:0.97451
[500]	validation_0-mlogloss:0.07655	validation_0-bal_ACC:0.97539
[600]	validation_0-mlogloss:0.06817	validation_0-bal_ACC:0.97610
[700]	validation_0-mlogloss:0.06324	validation_0-bal_ACC:0.97649
[800]	validation_0-mlogloss:0.06017	validation_0-bal_ACC:0.97650
[900]	validation_0-mlogloss:0.05802	validation_0-bal_ACC:0.97656
[1000]	validation_0-mlogloss:0.05635	validation_0-bal_ACC:0.97668
[1100]	validation_0-mlogloss:0.05512	validation_0-bal_ACC:0.97664
[1200]	validation_0-mlogloss:0.05415	validation_0-bal_ACC:0.97673
[1300]	validation_0-mlogloss:0.05339	validation_0-bal_ACC:0.97690
[1400]	validation_0-mlogloss:0.05276	validation_0-bal_ACC:0.97

In [6]:
submission = pd.read_csv('/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv')
submission[TARGET] = np.argmax(pred, axis=1)
submission[TARGET] = submission[TARGET].map({0: 'Low', 1: 'Medium', 2: 'High'})
submission.to_csv('xgb.csv', index=False)
pd.DataFrame({'xgb_oof': oof.flatten()}).to_csv('xgb_oof.csv', index=False)
pd.DataFrame({'xgb_pred': pred.flatten()}).to_csv('xgb_pred.csv', index=False)